# Import libraries & set constants

## Import Libraries

In [1]:
!pip install ccxt pandas mplfinance
!pip install boto3
!pip install numba

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.8/151.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.8/223.8 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 117.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 7.0 MB/s eta 0:00:00


In [2]:
import ccxt
import pandas as pd
from datetime import datetime
import time
import io
import math
from google.colab import drive
import os
import sys
import boto3
from google.colab import userdata

import numpy as np
from scipy.signal import find_peaks
from tqdm import tqdm
from tqdm.auto import tqdm
from tqdm.notebook import tqdm
import random
import mplfinance as mpf
from scipy.stats import linregress
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader
import xgboost as xgb
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
# ⏪ [AI REVERT]: Removed RandomForestClassifier import (reverting two-stage cascade)
import time
from sklearn.metrics import classification_report, confusion_matrix, roc_curve
from sklearn.metrics import auc, precision_recall_curve, average_precision_score, make_scorer
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import make_scorer, fbeta_score
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import TimeSeriesSplit
import joblib
import json
import subprocess
import plotly.graph_objects as go
from numba import njit
import concurrent.futures
import matplotlib
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
matplotlib.use('Agg')

drive.mount('/content/drive')

Mounted at /content/drive


## Set constants

In [3]:
symbols = [
    'BTC/USDT', 'ETH/USDT', 'SOL/USDT', 'BNB/USDT', 'XRP/USDT',
    'ADA/USDT', 'DOT/USDT', 'LINK/USDT', 'AVAX/USDT', 'DOGE/USDT',
    'NEAR/USDT', 'ATOM/USDT', 'LTC/USDT', 'PEPE/USDT',
    'SHIB/USDT', 'FET/USDT', 'SUI/USDT', 'APT/USDT', 'OP/USDT',
    'ARB/USDT', 'RENDER/USDT', 'INJ/USDT', 'TIA/USDT'
]
pattern_length = 50
cooldown_jump = int(pattern_length * 0.33)
normal_step = 2

In [4]:
def get_optimal_device():
    """
    Проверяет наличие видеокарты NVIDIA в системе.
    Возвращает 'cuda' если доступно, иначе 'cpu'.
    """
    try:
        # Пытаемся вызвать системную утилиту драйвера видеокарты
        subprocess.check_output('nvidia-smi')
        print("🟢 Ура! Обнаружен GPU (CUDA). Включаем турборежим.")
        return 'cuda'
    except Exception:
        print("🟡 GPU не найден или недоступен. Откатываемся на CPU (процессор).")
        return 'cpu'

# Определяем устройство для всей дальнейшей работы
ACTIVE_DEVICE = get_optimal_device()

🟡 GPU не найден или недоступен. Откатываемся на CPU (процессор).


In [5]:
TARGET_MAP = {
    "NOISE": 0,
    "DT": 1,
    "DB": 2
}

In [6]:
YANDEX_S3_BUCKET = 'tickframe-candidates'
ACCESS_KEY_ID = userdata.get("ACCESS_KEY_ID")
SECRET_ACCESS_KEY = userdata.get("SECRET_ACCESS_KEY")
sample_patterns_prefix = "evaluation_sample"
main_dataset_prefix = "labeled_train_v1"

# Auto-data-labeling

## Pooling raw 5-minutes candles from google drive

In [ ]:
def isSymbolInFileName(symbols, filename):
    for symbol in symbols:
        if symbol.partition('/')[0] in filename:
            return True
    return False

In [ ]:
raw_input_folder = '/content/drive/MyDrive/Crypto_Raw_Data'
raw_candles = {}

print("Downloading raw data from disk")
for symbol in symbols:
    clean_name = symbol.replace('/', '_') + '_raw.csv'
    path = f'{raw_input_folder}/{clean_name}'
    if os.path.exists(path):
        raw_candles[symbol] = pd.read_csv(path, index_col=0, parse_dates=True)
        print(f"Data: {symbol} ({len(raw_candles[symbol])} строк)")

Data: BTC/USDT (888066 строк)
Data: ETH/USDT (888061 строк)
Data: SOL/USDT (577029 строк)
Data: BNB/USDT (473350 строк)
Data: XRP/USDT (888071 строк)
Data: ADA/USDT (853513 строк)
Data: DOT/USDT (602954 строк)
Data: LINK/USDT (784396 строк)
Data: AVAX/USDT (594317 строк)
Data: DOGE/USDT (749838 строк)
Data: NEAR/USDT (594320 строк)
Data: ATOM/USDT (749842 строк)
Data: LTC/USDT (888083 строк)
Data: PEPE/USDT (326484 строк)
Data: SHIB/USDT (533845 строк)
Data: FET/USDT (343766 строк)
Data: SUI/USDT (326486 строк)
Data: APT/USDT (378327 строк)
Data: OP/USDT (421527 строк)
Data: ARB/USDT (335128 строк)
Data: RENDER/USDT (196888 строк)
Data: INJ/USDT (559769 строк)
Data: TIA/USDT (274601 строк)


In [ ]:
display(raw_candles[symbols[0]].head())

,Open,High,Low,Close,Volume
Datetime,,,,,
2018-01-01 00:00:00,13704.00,13708.57,13679.42,13680.00,20.503984
2018-01-01 00:05:00,13679.42,13680.00,13616.54,13619.03,52.307490
2018-01-01 00:10:00,13619.03,13621.38,13593.16,13600.02,39.299696
2018-01-01 00:15:00,13599.97,13603.54,13519.66,13519.81,43.473560
2018-01-01 00:20:00,13519.66,13564.21,13501.87,13502.92,43.954285


## Add smart features to candles

In [ ]:
@njit
def _find_extrema_numba(high_windows, low_windows, window_size, min_dist):
    """
    JIT-скомпилированная функция для молниеносного поиска 2-х пиков и 2-х впадин.
    """
    n_windows = len(high_windows)

    macro_high_indices = np.zeros((n_windows, 2))
    macro_high_prices = np.zeros((n_windows, 2))
    macro_low_indices = np.zeros((n_windows, 2))
    macro_low_prices = np.zeros((n_windows, 2))

    for i in range(n_windows):
        h_win = high_windows[i]
        l_win = low_windows[i]

        available_h = np.ones(window_size, dtype=np.bool_)
        available_l = np.ones(window_size, dtype=np.bool_)

        # Ищем 2 пика
        for step in range(2):
            best_h_val = -np.inf
            best_h_idx = -1

            # Вручную находим максимум с учетом доступности
            for j in range(window_size):
                if available_h[j] and h_win[j] > best_h_val:
                    best_h_val = h_win[j]
                    best_h_idx = j

            if best_h_idx == -1:
                break

            macro_high_indices[i, step] = window_size - best_h_idx
            macro_high_prices[i, step] = best_h_val

            # Закрашиваем область вокруг найденного пика (выключаем)
            start_idx_h = max(0, best_h_idx - min_dist)
            end_idx_h = min(window_size, best_h_idx + min_dist + 1)
            for j in range(start_idx_h, end_idx_h):
                available_h[j] = False

        # Ищем 2 впадины
        for step in range(2):
            best_l_val = np.inf
            best_l_idx = -1

            # Вручную находим минимум с учетом доступности
            for j in range(window_size):
                if available_l[j] and l_win[j] < best_l_val:
                    best_l_val = l_win[j]
                    best_l_idx = j

            if best_l_idx == -1:
                break

            macro_low_indices[i, step] = window_size - best_l_idx
            macro_low_prices[i, step] = best_l_val

            # Закрашиваем область вокруг найденной впадины
            start_idx_l = max(0, best_l_idx - min_dist)
            end_idx_l = min(window_size, best_l_idx + min_dist + 1)
            for j in range(start_idx_l, end_idx_l):
                available_l[j] = False

    return macro_high_indices, macro_high_prices, macro_low_indices, macro_low_prices

In [ ]:
def add_smart_features_dtdb(df, window_size=50):
    """
    УЛЬТРА-БЫСТРАЯ ВЕРСИЯ: Извлекает геометрию для паттернов Double Top (DT) и Double Bottom (DB).
    Использует Numba JIT для поиска экстремумов.
    """
    # Создаем общий прогресс-бар для этой функции
    with tqdm(total=5, desc="  ↳ Извлечение Smart Features", leave=False) as pbar:
        data = df.copy()

        # =========================================================================
        # 1. БАЗОВЫЕ МЕТРИКИ (ATR и Контекст рынка)
        # =========================================================================
        w_natr = max(5, int(window_size * 0.28))
        min_dist = max(2, window_size // 10)

        prev_close = data['Close'].shift(1)
        true_range = pd.concat([
            data['High'] - data['Low'],
            abs(data['High'] - prev_close),
            abs(data['Low'] - prev_close),
        ], axis=1).max(axis=1)
        data[f'NATR_{w_natr}'] = true_range.rolling(w_natr).mean() / data['Close']

        data[f'Trend_{window_size}'] = data['Close'] / data['Close'].shift(window_size) - 1
        min_window = data['Low'].rolling(window_size).min()
        max_window = data['High'].rolling(window_size).max()
        data['Range_Position'] = (data['Close'] - min_window) / (max_window - min_window + 1e-8)

        pbar.update(1) # Шаг 1 выполнен

        # =========================================================================
        # 2. ВЕКТОРНЫЙ ПОИСК ЭКСТРЕМУМОВ (ЧЕРЕЗ NUMBA)
        # =========================================================================
        high_prices = data['High'].values
        low_prices = data['Low'].values

        high_windows = sliding_window_view(high_prices, window_shape=window_size)
        low_windows = sliding_window_view(low_prices, window_shape=window_size)

        # Вызываем скомпилированную функцию (первый вызов займет ~1 сек на компиляцию, остальные пролетят)
        macro_high_indices, macro_high_prices, macro_low_indices, macro_low_prices = \
            _find_extrema_numba(high_windows, low_windows, window_size, min_dist)

        pbar.update(1) # Шаг 2 выполнен

        # =========================================================================
        # 3. ХРОНОЛОГИЧЕСКАЯ СОРТИРОВКА (1=Левый пик, 2=Правый пик)
        # =========================================================================
        sort_idx_h = np.argsort(-macro_high_indices, axis=1)
        macro_high_indices = np.take_along_axis(macro_high_indices, sort_idx_h, axis=1)
        macro_high_prices = np.take_along_axis(macro_high_prices, sort_idx_h, axis=1)

        sort_idx_l = np.argsort(-macro_low_indices, axis=1)
        macro_low_indices = np.take_along_axis(macro_low_indices, sort_idx_l, axis=1)
        macro_low_prices = np.take_along_axis(macro_low_prices, sort_idx_l, axis=1)

        pbar.update(1) # Шаг 3 выполнен

        # =========================================================================
        # 4. ВЫЧИСЛЕНИЕ НОРМАЛИЗОВАННЫХ КООРДИНАТ С УЧЕТОМ ATR
        # =========================================================================
        data = data.iloc[window_size - 1:].copy()
        current_closes = data['Close'].values
        current_atr_usd = data[f'NATR_{w_natr}'].values * current_closes + 1e-8

        for step in range(2):
            data[f'H_Idx_{step+1}'] = macro_high_indices[:, step].astype(int)
            data[f'L_Idx_{step+1}'] = macro_low_indices[:, step].astype(int)

            data[f'H_Prc_{step+1}'] = (macro_high_prices[:, step] - current_closes) / current_atr_usd
            data[f'L_Prc_{step+1}'] = (current_closes - macro_low_prices[:, step]) / current_atr_usd

        pbar.update(1) # Шаг 4 выполнен

        # =========================================================================
        # 5. ОЦИФРОВКА ЭСТЕТИКИ DOUBLE TOP / DOUBLE BOTTOM
        # =========================================================================
        data['DT_Width'] = data['H_Idx_1'] - data['H_Idx_2']
        data['DB_Width'] = data['L_Idx_1'] - data['L_Idx_2']

        data['DT_Symmetry_Prc'] = abs(data['H_Prc_1'] - data['H_Prc_2'])
        data['DB_Symmetry_Prc'] = abs(data['L_Prc_1'] - data['L_Prc_2'])

        pbar.update(1) # Шаг 5 выполнен

        return data

In [ ]:
featured_data = {}
for symbol in tqdm(symbols):
    featured_data[symbol] = add_smart_features_dtdb(raw_candles[symbol], window_size=50)
display(featured_data[symbols[0]])

  0%|          | 0/23 [00:00<?, ?it/s]

  ↳ Извлечение Smart Features:   0%|          | 0/5 [00:00<?, ?it/s]

  ↳ Извлечение Smart Features:   0%|          | 0/5 [00:00<?, ?it/s]

  ↳ Извлечение Smart Features:   0%|          | 0/5 [00:00<?, ?it/s]

  ↳ Извлечение Smart Features:   0%|          | 0/5 [00:00<?, ?it/s]

...

  ↳ Извлечение Smart Features:   0%|          | 0/5 [00:00<?, ?it/s]

  ↳ Извлечение Smart Features:   0%|          | 0/5 [00:00<?, ?it/s]

  ↳ Извлечение Smart Features:   0%|          | 0/5 [00:00<?, ?it/s]

  ↳ Извлечение Smart Features:   0%|          | 0/5 [00:00<?, ?it/s]

,Open,High,Low,Close,Volume,NATR_14,Trend_50,Range_Position,H_Idx_1,L_Idx_1,H_Prc_1,L_Prc_1,H_Idx_2,L_Idx_2,H_Prc_2,L_Prc_2,DT_Width,DB_Width,DT_Symmetry_Prc,DB_Symmetry_Prc
Datetime,,,,,,,,,,,,,,,,,,,,
2018-01-01 04:05:00,13409.34,13438.05,13382.81,13382.81,33.316957,0.005811,NaN,0.346611,50,27,4.189108,2.222249,41,18,3.451616,2.034114,9,9,0.737492,0.188134
2018-01-01 04:10:00,13391.72,13419.94,13326.81,13391.91,26.848109,0.005877,-0.021059,0.387043,50,28,3.660543,2.311394,42,19,3.294858,2.125501,8,9,0.365686,0.185892
2018-01-01 04:15:00,13371.10,13391.87,13315.99,13341.54,21.888311,0.005851,-0.020375,0.298128,43,29,3.967023,1.685037,35,20,3.627428,1.497626,8,9,0.339595,0.187411
2018-01-01 04:20:00,13348.73,13433.47,13320.62,13410.00,23.512795,0.005625,-0.013972,0.453289,44,30,3.198053,2.651565,36,21,2.846588,2.457603,8,9,0.351465,0.193962
2018-01-01 04:25:00,13410.00,13498.00,13380.91,13486.99,29.719187,0.005908,-0.002428,0.627782,45,31,2.061027,3.476124,37,22,1.728336,3.292523,8,9,0.332691,0.183601
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-11 13:05:00,62931.99,62931.99,62882.22,62885.53,2.932972,0.002369,-0.000966,0.483055,26,45,2.221779,0.626365,20,8,2.544527,2.377716,6,37,0.322748,1.751351
2026-06-11 13:10:00,62885.53,62919.37,62844.30,62894.35,3.653969,0.002346,-0.000487,0.495084,27,46,2.183247,0.692125,21,9,2.509081,2.460220,6,37,0.325834,1.768094
2026-06-11 13:15:00,62902.44,62957.95,62890.90,62957.95,0.741445,0.002314,0.001754,0.581818,28,47,1.775162,1.137831,22,10,2.105259,2.929059,6,37,0.330097,1.791228


## Labeling potential candidates

In [ ]:
def label_dt_db_candidates(df, window_size=50, atr_threshold=0.3, min_width=10, max_width=30, dip_atr_min=2.5, min_dip_pct=0.012, noise_space=0.5):
    """
    РАЗМЕТКА 'GOLDILOCKS' (Target: ~3,500 candidates).
    Идеальный баланс: провал 1.2% + 2.5 ATR, и строгий контроль начала тренда.
    """
    data = df.copy()
    data['Target'] = TARGET_MAP["NOISE"]

    NOISE, DT, DB = TARGET_MAP["NOISE"], TARGET_MAP["DT"], TARGET_MAP["DB"]

    lows = data['Low'].values
    highs = data['High'].values
    closes = data['Close'].values
    natr = data['NATR_14'].values

    h_idx_1 = data['H_Idx_1'].values.astype(int)
    h_idx_2 = data['H_Idx_2'].values.astype(int)
    dt_width = data['DT_Width'].values
    dt_sym = data['DT_Symmetry_Prc'].values

    l_idx_1 = data['L_Idx_1'].values.astype(int)
    l_idx_2 = data['L_Idx_2'].values.astype(int)
    db_width = data['DB_Width'].values
    db_sym = data['DB_Symmetry_Prc'].values

    for i in tqdm(range(len(data)), desc="  ↳ Разметка (Goldilocks)", leave=False):
        if i < window_size + 20:
            continue

        current_atr_usd = natr[i] * closes[i]

        # ==========================================
        # DOUBLE TOP (DT)
        # ==========================================
        if (min_width <= dt_width[i] <= max_width) and (dt_sym[i] <= atr_threshold):

            if h_idx_1[i] >= (window_size - 3):
                continue

            idx_left_peak = i - h_idx_1[i]
            idx_right_peak = i - h_idx_2[i]

            if idx_left_peak >= 0 and idx_left_peak < idx_right_peak:
                valley_low = np.min(lows[idx_left_peak : idx_right_peak])
                avg_peak_high = (highs[idx_left_peak] + highs[idx_right_peak]) / 2
                dip_depth_usd = avg_peak_high - valley_low

                # 1. ЗАЩИТА ОТ ШУМА: Строгий гибрид (2.5 ATR и 1.2% от цены)
                if dip_depth_usd > (dip_atr_min * current_atr_usd) and (dip_depth_usd / closes[i]) >= min_dip_pct:

                    # 2. ФИЛЬТР ВОЗДУХА: Строго 50% долины должно быть пустым
                    highs_between = highs[idx_left_peak+1 : idx_right_peak]
                    if len(highs_between) > 0:
                        if np.mean(highs_between) > (avg_peak_high - dip_depth_usd * (1 - noise_space)):
                            continue

                    # 3. МАКРО-ТРЕНД: Проверяем НАЧАЛО тренда (база из 3 свечей за 20 баров до пика)
                    pre_start = max(0, idx_left_peak - 20)
                    trend_base_avg = np.mean(closes[pre_start : pre_start+3])

                    if trend_base_avg < valley_low:

                        # 4. ТРИГГЕР: Падение на 0.75 ATR от правого пика
                        if closes[i] < highs[idx_right_peak] - (current_atr_usd * 0.75):
                            data.at[data.index[i], 'Target'] = DT
                            continue

        # ==========================================
        # DOUBLE BOTTOM (DB)
        # ==========================================
        if (min_width <= db_width[i] <= max_width) and (db_sym[i] <= atr_threshold):

            if l_idx_1[i] >= (window_size - 3):
                continue

            idx_left_valley = i - l_idx_1[i]
            idx_right_valley = i - l_idx_2[i]

            if idx_left_valley >= 0 and idx_left_valley < idx_right_valley:
                peak_high = np.max(highs[idx_left_valley : idx_right_valley])
                avg_valley_low = (lows[idx_left_valley] + lows[idx_right_valley]) / 2
                rise_height_usd = peak_high - avg_valley_low

                # 1. ЗАЩИТА ОТ ШУМА
                if rise_height_usd > (dip_atr_min * current_atr_usd) and (rise_height_usd / closes[i]) >= min_dip_pct:

                    # 2. ФИЛЬТР ВОЗДУХА
                    lows_between = lows[idx_left_valley+1 : idx_right_valley]
                    if len(lows_between) > 0:
                        if np.mean(lows_between) < (avg_valley_low + rise_height_usd * (1 - noise_space)):
                            continue

                    # 3. МАКРО-ТРЕНД: Проверяем НАЧАЛО падения
                    pre_start = max(0, idx_left_valley - 20)
                    trend_base_avg = np.mean(closes[pre_start : pre_start+3])

                    if trend_base_avg > peak_high:

                        # 4. ТРИГГЕР: Рост на 0.75 ATR от правого дна
                        if closes[i] > lows[idx_right_valley] + (current_atr_usd * 0.75):
                            data.at[data.index[i], 'Target'] = DB

    return data

In [ ]:
def apply_true_nms(df, distance_threshold=10):
    """
    УЛЬТРА-ФАСТ ВЕРСИЯ NMS НА NUMPY.
    Работает на основе вектора подавления (Boolean Mask).
    Снижает время обработки с часов до долей секунды.
    """
    data = df.copy()

    # 1. Мгновенно вытаскиваем данные в чистые массивы C-уровня (убираем оверхед .iloc)
    targets = data['Target'].values
    dt_sym = data['DT_Symmetry_Prc'].values
    db_sym = data['DB_Symmetry_Prc'].values

    # Находим индексы всех строк, где эвристика нашла хоть какой-то паттерн
    candidate_indices = np.where(targets != TARGET_MAP["NOISE"])[0]

    if len(candidate_indices) == 0:
        return data

    # 2. Векторно собираем "скоры" (симметрию) для каждого кандидата
    candidate_targets = targets[candidate_indices]
    candidate_scores = np.where(
        candidate_targets == TARGET_MAP["DT"],
        dt_sym[candidate_indices],
        db_sym[candidate_indices]
    )

    # 3. Сортируем индексы по возрастанию score (лучшая симметрия уходит в начало)
    sort_idx = np.argsort(candidate_scores)
    sorted_indices = candidate_indices[sort_idx]
    sorted_targets = candidate_targets[sort_idx]

    # 4. Создаем маску подавления (True там, где паттерн уже "задавлен" более сильным соседом)
    is_suppressed = np.zeros(len(data), dtype=bool)

    winners_indices = []
    winners_targets = []

    # 5. Жадное подавление (работает со скоростью света)
    for idx, target in zip(sorted_indices, sorted_targets):
        # Если эта свеча уже попала в радиус подавления более красивого паттерна — скипаем её
        if is_suppressed[idx]:
            continue

        # Если не подавлен — перед нами локальный геометрический пик (победитель)
        winners_indices.append(idx)
        winners_targets.append(target)

        # Векторно за секунду "выжигаем" всех соседей в радиусе вокруг победителя
        start = max(0, idx - distance_threshold)
        end = min(len(data), idx + distance_threshold + 1)
        is_suppressed[start:end] = True

    # 6. Формируем финальный очищенный вектор таргет-классов
    new_targets = np.zeros(len(data), dtype=int)
    if winners_indices:
        new_targets[winners_indices] = winners_targets

    data['Target'] = new_targets
    return data

In [ ]:
labeled_data = {}
for symbol in tqdm(symbols, desc = "Labeling candidates"):
    labeled_data[symbol] = label_dt_db_candidates(featured_data[symbol],
                                                                 window_size=pattern_length,
                                                   atr_threshold=0.4, #symmetry
                                                   min_width=8,
                                                   max_width=30,
                                                   dip_atr_min=1.7, #depth
                                                   min_dip_pct=0.010, #depth in absolute val
                                                   noise_space=0.6 #shadow flexibility
                                                   )

Labeling candidates:   0%|          | 0/23 [00:00<?, ?it/s]

  ↳ Разметка (Goldilocks):   0%|          | 0/888017 [00:00<?, ?it/s]

  ↳ Разметка (Goldilocks):   0%|          | 0/888012 [00:00<?, ?it/s]

  ↳ Разметка (Goldilocks):   0%|          | 0/576980 [00:00<?, ?it/s]

  ↳ Разметка (Goldilocks):   0%|          | 0/473301 [00:00<?, ?it/s]

...

  ↳ Разметка (Goldilocks):   0%|          | 0/421478 [00:00<?, ?it/s]

  ↳ Разметка (Goldilocks):   0%|          | 0/335079 [00:00<?, ?it/s]

  ↳ Разметка (Goldilocks):   0%|          | 0/196839 [00:00<?, ?it/s]

  ↳ Разметка (Goldilocks):   0%|          | 0/559720 [00:00<?, ?it/s]

  ↳ Разметка (Goldilocks):   0%|          | 0/274552 [00:00<?, ?it/s]

In [ ]:
clear_labeled_data = {}
for symbol in tqdm(symbols, desc="Labeling Candidates"):
    clear_labeled_data[symbol] = apply_true_nms(labeled_data[symbol], distance_threshold=7)

Labeling Candidates:   0%|          | 0/23 [00:00<?, ?it/s]

In [ ]:
total_dt = 0
total_db = 0

print("=== СТАТИСТИКА РАЗМЕТКИ ПОСЛЕ NMS ===")
for symbol, df in clear_labeled_data.items():
    counts = df['Target'].value_counts()

    dt_count = counts.get(TARGET_MAP["DT"], 0)
    db_count = counts.get(TARGET_MAP["DB"], 0)

    total_dt += dt_count
    total_db += db_count

    print(f"[{symbol}] Double Tops: {dt_count} | Double Bottoms: {db_count}")

print("-" * 37)
print(f"ВСЕГО НАЙДЕНО -> DT: {total_dt} | DB: {total_db} | total: {total_dt + total_db}")
candles_cnt = sum(len(raw_candles[symbol]) for symbol in symbols)
print(f"Generally it covers (rude count): {(total_dt + total_db) * 50} candles ({(total_dt + total_db) * 50 / candles_cnt * 100 : .2f}%)")
print(f"The amount of candles is {candles_cnt}")

=== СТАТИСТИКА РАЗМЕТКИ ПОСЛЕ NMS ===
[BTC/USDT] Double Tops: 22 | Double Bottoms: 68
[ETH/USDT] Double Tops: 83 | Double Bottoms: 111
[SOL/USDT] Double Tops: 93 | Double Bottoms: 111
[BNB/USDT] Double Tops: 28 | Double Bottoms: 55
[XRP/USDT] Double Tops: 99 | Double Bottoms: 94
[ADA/USDT] Double Tops: 161 | Double Bottoms: 172
[DOT/USDT] Double Tops: 118 | Double Bottoms: 137
[LINK/USDT] Double Tops: 174 | Double Bottoms: 180
[AVAX/USDT] Double Tops: 91 | Double Bottoms: 154
[DOGE/USDT] Double Tops: 78 | Double Bottoms: 130
[NEAR/USDT] Double Tops: 152 | Double Bottoms: 144
[ATOM/USDT] Double Tops: 165 | Double Bottoms: 165
[LTC/USDT] Double Tops: 139 | Double Bottoms: 161
[PEPE/USDT] Double Tops: 108 | Double Bottoms: 124
[SHIB/USDT] Double Tops: 56 | Double Bottoms: 94
[FET/USDT] Double Tops: 88 | Double Bottoms: 88
[SUI/USDT] Double Tops: 59 | Double Bottoms: 57
[APT/USDT] Double Tops: 75 | Double Bottoms: 68
[OP/USDT] Double Tops: 114 | Double Bottoms: 74
[ARB/USDT] Double Tops: 5

## Output potential candidates

In [ ]:
def plot_dtdb_candidates_style(labeled_dict, target_class, num_charts=3, window_size=50):
    """
    Рисует свечные графики кандидатов DT/DB в фирменном стиле (Matplotlib).
    Использует ТОЛЬКО словарь clear_labeled_data.
    """
    # 1. Собираем глобальный пул всех индексов паттернов со всех монет
    pool = []
    for symbol, df in labeled_dict.items():
        # Временно сбрасываем индекс, чтобы работать с числовыми позициями строк
        temp_df = df.reset_index()
        matches = temp_df[temp_df['Target'] == target_class].index.tolist()
        for idx in matches:
            pool.append((symbol, idx))

    if not pool:
        print(f"⚠️ Во всем словаре не найдено паттернов класса {target_class}.")
        return

    # 2. Выбираем случайные паттерны
    dynamic_seed = int(time.time() * 1000) % (2**32 - 1)
    rng = np.random.default_rng(dynamic_seed)

    # Защита от случая, если кандидатов меньше, чем num_charts
    sample_size = min(num_charts, len(pool))
    chosen = [pool[i] for i in rng.choice(len(pool), size=sample_size, replace=False)]

    class_map = {0: "Noise", 1: "Double Top", 2: "Double Bottom"}
    padding = int(window_size * 0.33)

    for symbol, pos in chosen:
        # Достаем датафрейм монеты
        df = labeled_dict[symbol].reset_index()
        y_true_val = df.loc[pos, 'Target']

        # Заглушки для сохранения стиля заголовка (пока нет модели)
        y_pred_val = y_true_val
        probas = [0.11, 0.89, 0.00] if y_true_val == 1 else [0.05, 0.05, 0.90]

        # Формируем границы среза
        start_pos = max(0, pos - window_size - padding)
        end_pos = min(len(df), pos + padding + 1)
        vis_df = df.iloc[start_pos:end_pos].copy()

        fig, ax = plt.subplots(figsize=(12, 5))
        x_vals = np.arange(len(vis_df))

        trigger_x = pos - start_pos
        pattern_start_x = trigger_x - window_size

        # Определяем колонки OHLC
        cols = [c.lower() for c in vis_df.columns]
        o_col, h_col = vis_df.columns[cols.index('open')], vis_df.columns[cols.index('high')]
        l_col, c_col = vis_df.columns[cols.index('low')], vis_df.columns[cols.index('close')]

        up = vis_df[vis_df[c_col] >= vis_df[o_col]]
        down = vis_df[vis_df[c_col] < vis_df[o_col]]

        # Зеленые свечи
        ax.vlines(x_vals[vis_df[c_col] >= vis_df[o_col]], up[l_col], up[h_col], color='#26A69A', linewidth=1.5)
        ax.bar(x_vals[vis_df[c_col] >= vis_df[o_col]], up[c_col] - up[o_col], bottom=up[o_col], color='#26A69A', width=0.7)

        # Красные свечи
        ax.vlines(x_vals[vis_df[c_col] < vis_df[o_col]], down[l_col], down[h_col], color='#EF5350', linewidth=1.5)
        ax.bar(x_vals[vis_df[c_col] < vis_df[o_col]], down[o_col] - down[c_col], bottom=down[c_col], color='#EF5350', width=0.7)

        trigger_y = vis_df.iloc[trigger_x][c_col]

        # 1. Красные пунктирные рамки сканирования
        ax.axvline(pattern_start_x, color="red", linestyle="--", linewidth=2, alpha=0.7)
        ax.axvline(trigger_x, color="red", linestyle="--", linewidth=2, alpha=0.7)

        # 2. Легкая заливка зоны паттерна
        bg_color = "red" if y_pred_val == 1 else "green"
        ax.axvspan(pattern_start_x, trigger_x, color=bg_color, alpha=0.05)

        # 3. Синяя точка (момент принятия решения моделью)
        if 0 <= trigger_x < len(vis_df):
            ax.scatter(trigger_x, trigger_y, color="blue", s=100, zorder=5, edgecolors='black', label="Сигнал алгоритма")

        # Настраиваем красивые метки времени (по оси X)
        step = max(1, len(vis_df) // 10)
        ax.set_xticks(x_vals[::step])
        labels = []

        # Ищем колонку со временем
        x_col = 'Datetime' if 'Datetime' in vis_df.columns else ('timestamp' if 'timestamp' in vis_df.columns else 'index')

        for i in range(0, len(vis_df), step):
            if x_col in vis_df.columns:
                idx_val = vis_df.iloc[i][x_col]
            else:
                idx_val = vis_df.index[i]
            labels.append(idx_val.strftime('%m-%d %H:%M') if hasattr(idx_val, 'strftime') else str(idx_val))

        ax.set_xticklabels(labels, rotation=15, ha='right', fontsize=9)

        # Форматируем заголовок (Точь-в-точь как на скрине)
        probas_str = "[" + ", ".join(f"{p:.2f}" for p in probas) + "]"
        title = f"REAL Candidate | Idx={pos} | {symbol}\nTrue={class_map[y_true_val]} -> Pred={class_map[y_pred_val]}\nProbs={probas_str}"

        ax.set_title(title, fontsize=11)
        ax.grid(alpha=0.3)
        ax.legend(loc="upper left")
        plt.tight_layout()
        plt.show()

In [ ]:
# Построить 5 графиков Double Top
plot_dtdb_candidates_style(
    labeled_dict=clear_labeled_data,
    target_class=1,  # TARGET_MAP["DT"]
    num_charts=10,
    window_size=50
)

# Построить 5 графиков Double Bottom
plot_dtdb_candidates_style(
    labeled_dict=clear_labeled_data,
    target_class=2,  # TARGET_MAP["DB"]
    num_charts=10,
    window_size=50
)

# Functions for working with DB

In [ ]:
def upload_heuristic_sample_to_s3(labeled_dict, s3_client, bucket_name, s3_folder, target_type, sample_size=200, window_size=50, max_workers=15):
    """
    Потокобезопасная загрузка графиков в S3.
    target_type: может быть числом (например, 1) или списком (например, [1, 2]).
    sample_size: число (200) или "all" / None для выгрузки всех.
    """

    # 1. Умная обработка входного типа (превращаем в список, если передано одно число)
    if isinstance(target_type, (int, float)):
        target_types = [int(target_type)]
    else:
        target_types = list(target_type)

    # Словарь-помощник для названий и префиксов
    meta_map = {
        1: {"name": "Double Top", "prefix": "DT", "color": "red"},
        2: {"name": "Double Bottom", "prefix": "DB", "color": "green"}
    }

    types_str = " & ".join([meta_map.get(t, {}).get("name", f"Type_{t}") for t in target_types])
    print(f"☁️ Подготовка [{types_str}] к загрузке в S3 (Бакет: {bucket_name}, Папка: {s3_folder})")

    # 2. Собираем глобальный пул кандидатов (теперь сохраняем и тип таргета!)
    pool = []
    for symbol, df in labeled_dict.items():
        temp_df = df.reset_index()
        # Фильтруем сразу по нескольким таргетам через .isin()
        matches = temp_df[temp_df['Target'].isin(target_types)].index.tolist()
        for idx in matches:
            tgt = int(temp_df.loc[idx, 'Target'])
            pool.append((symbol, idx, tgt)) # <-- Добавили tgt в кортеж

    if not pool:
        print(f"⚠️ Не найдено ни одного паттерна для отправки.")
        return

    # 3. Логика выборки (Sample или ВСЕ)
    if sample_size is None or str(sample_size).lower() == "all":
        print(f"📊 Выбрана загрузка АБСОЛЮТНО ВСЕХ кандидатов ({len(pool)} шт.)...")
        chosen_samples = pool
    else:
        actual_size = min(sample_size, len(pool))
        print(f"📊 Всего найдено: {len(pool)}. Случайная выборка: {actual_size} шт...")
        dynamic_seed = int(time.time() * 1000) % (2**32 - 1)
        rng = np.random.default_rng(dynamic_seed)
        chosen_samples = [pool[i] for i in rng.choice(len(pool), size=actual_size, replace=False)]

    padding = int(window_size * 0.33)

    # 4. ВОРКЕР (Рисует и отправляет 1 картинку)
    def process_and_upload(sample):
        # Теперь мы распаковываем ТРИ значения, включая таргет
        symbol, idx, target_val = sample
        df = labeled_dict[symbol].reset_index()

        # Получаем метаданные для конкретно этого графика
        t_meta = meta_map.get(target_val, {"name": "Unknown", "prefix": "UNK", "color": "gray"})
        pattern_name = t_meta["name"]
        prefix_name = t_meta["prefix"]
        bg_color = t_meta["color"]

        start_pos = max(0, idx - window_size - padding)
        end_pos = min(len(df), idx + padding + 1)
        vis_df = df.iloc[start_pos:end_pos].copy()

        if vis_df.empty:
            return False

        fig = Figure(figsize=(12, 5))
        canvas = FigureCanvasAgg(fig)
        ax = fig.add_subplot(111)

        x_vals = np.arange(len(vis_df))
        trigger_x = idx - start_pos
        pattern_start_x = trigger_x - window_size

        cols = [c.lower() for c in vis_df.columns]
        o_col, h_col = vis_df.columns[cols.index('open')], vis_df.columns[cols.index('high')]
        l_col, c_col = vis_df.columns[cols.index('low')], vis_df.columns[cols.index('close')]

        up = vis_df[vis_df[c_col] >= vis_df[o_col]]
        down = vis_df[vis_df[c_col] < vis_df[o_col]]

        ax.vlines(x_vals[vis_df[c_col] >= vis_df[o_col]], up[l_col], up[h_col], color='#26A69A', linewidth=1.5)
        ax.bar(x_vals[vis_df[c_col] >= vis_df[o_col]], up[c_col] - up[o_col], bottom=up[o_col], color='#26A69A', width=0.7)
        ax.vlines(x_vals[vis_df[c_col] < vis_df[o_col]], down[l_col], down[h_col], color='#EF5350', linewidth=1.5)
        ax.bar(x_vals[vis_df[c_col] < vis_df[o_col]], down[o_col] - down[c_col], bottom=down[c_col], color='#EF5350', width=0.7)

        trigger_y = vis_df.iloc[trigger_x][c_col]

        ax.axvline(pattern_start_x, color="red", linestyle="--", linewidth=2, alpha=0.7)
        ax.axvline(trigger_x, color="red", linestyle="--", linewidth=2, alpha=0.7)

        # Динамический цвет фона
        ax.axvspan(pattern_start_x, trigger_x, color=bg_color, alpha=0.05)

        if 0 <= trigger_x < len(vis_df):
            ax.scatter(trigger_x, trigger_y, color="blue", s=100, zorder=5, edgecolors='black', label="Сигнал алгоритма")

        step = max(1, len(vis_df) // 10)
        ax.set_xticks(x_vals[::step])
        labels = []
        x_col = 'Datetime' if 'Datetime' in vis_df.columns else ('timestamp' if 'timestamp' in vis_df.columns else 'index')
        for i in range(0, len(vis_df), step):
            idx_val = vis_df.iloc[i][x_col] if x_col in vis_df.columns else vis_df.index[i]
            labels.append(idx_val.strftime('%m-%d %H:%M') if hasattr(idx_val, 'strftime') else str(idx_val))
        ax.set_xticklabels(labels, rotation=15, ha='right', fontsize=9)

        # Динамический заголовок
        ax.set_title(f"Heuristic Evaluation | Idx={idx} | {symbol}\nAlgorithm Detected: {pattern_name}", fontsize=11)
        ax.grid(alpha=0.3)
        ax.legend(loc="upper left")

        fig.tight_layout()

        img_buffer = io.BytesIO()
        fig.savefig(img_buffer, format='png', dpi=100)
        img_buffer.seek(0)

        safe_symbol = symbol.replace("/", "_")

        # Динамическое имя файла
        filename = f"{s3_folder}/{prefix_name}_idx_{idx}_{safe_symbol}.png"

        s3_client.upload_fileobj(
            img_buffer,
            bucket_name,
            filename,
            ExtraArgs={'ContentType': 'image/png'}
        )

        del fig
        del canvas

        return True

    uploaded_count = 0
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = executor.map(process_and_upload, chosen_samples)
        for _ in tqdm(futures, total=len(chosen_samples), desc="Uploading Patterns"):
            uploaded_count += 1

    print(f"✅ Готово! Успешно загружено {uploaded_count} картинок в S3.")

In [ ]:
def clear_s3_folder_fast(s3_client, bucket_name, folder_prefix):
    """
    ПАКЕТНАЯ ОЧИСТКА: Удаляет до 1000 файлов за 1 сетевой запрос.
    Использует пагинацию для поддержки папок любого размера (даже > 1000 файлов).
    """
    if not folder_prefix.endswith('/'):
        folder_prefix += '/'

    print(f"🗑️ Начинаем очистку папки s3://{bucket_name}/{folder_prefix} ...")

    # Инициализируем пагинатор, чтобы прочитать ВСЕ файлы, сколько бы их ни было
    paginator = s3_client.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=bucket_name, Prefix=folder_prefix)

    deleted_count = 0

    # Создаем "бездонный" прогресс-бар, так как заранее не знаем точное количество файлов
    with tqdm(desc="  ↳ Удаление файлов", unit=" шт.") as pbar:
        for page in pages:
            if 'Contents' not in page:
                continue

            # 1. Формируем список ключей в формате, который требует boto3
            objects_to_delete = [{'Key': obj['Key']} for obj in page['Contents']]

            # 2. Удаляем весь батч (до 1000 штук) ОДНИМ запросом
            s3_client.delete_objects(
                Bucket=bucket_name,
                Delete={
                    'Objects': objects_to_delete,
                    'Quiet': True # Quiet=True ускоряет парсинг ответа от сервера
                }
            )

            # Обновляем счетчики
            batch_size = len(objects_to_delete)
            deleted_count += batch_size
            pbar.update(batch_size)

    if deleted_count == 0:
        print("🤷‍♂️ В папке пусто, удалять нечего!")
    else:
        print(f"✅ Успешно удалено файлов: {deleted_count} (Пакетное удаление)")

# Test labeling (with DataBase)

## Pushing candidates in DB for test hand labeling on Human signal

In [ ]:
s3_client = boto3.client(
    service_name='s3',
    endpoint_url='https://storage.yandexcloud.net',
    aws_access_key_id=ACCESS_KEY_ID,
    aws_secret_access_key=SECRET_ACCESS_KEY
)

In [ ]:
# 1. Отправляем 200 Double Tops (DT)
upload_heuristic_sample_to_s3(
    labeled_dict=clear_labeled_data,
    s3_client=s3_client,
    bucket_name=YANDEX_S3_BUCKET,
    s3_folder=sample_patterns_prefix,
    target_type=TARGET_MAP["DT"],   # 1 = DT
    sample_size=200,
    max_workers=5
)

# 2. Отправляем 200 Double Bottoms (DB)
upload_heuristic_sample_to_s3(
    labeled_dict=clear_labeled_data,
    s3_client=s3_client,
    bucket_name=YANDEX_S3_BUCKET,
    s3_folder=sample_patterns_prefix,
    target_type=TARGET_MAP["DB"],   # 2 = DB
    sample_size=200,
    max_workers=5
)

☁️ Подготовка Double Top к загрузке в S3 (Бакет: tickframe-candidates, Папка: evaluation_sample)
📊 Всего кандидатов Double Top в базе: 13256. Выбираем 200...


Uploading DT:   0%|          | 0/200 [00:00<?, ?it/s]

✅ Готово! Успешно загружено 200 картинок Double Top в S3.
☁️ Подготовка Double Bottom к загрузке в S3 (Бакет: tickframe-candidates, Папка: evaluation_sample)
📊 Всего кандидатов Double Bottom в базе: 14048. Выбираем 200...


Uploading DB:   0%|          | 0/200 [00:00<?, ?it/s]

✅ Готово! Успешно загружено 200 картинок Double Bottom в S3.


## Clear S3 backet on yandex cloud

In [ ]:
clear_s3_folder_fast(s3_client, YANDEX_S3_BUCKET, sample_patterns_prefix)

🗑️ Начинаем очистку папки s3://tickframe-candidates/evaluation_sample/ ...


  ↳ Удаление файлов: 0 шт. [00:00, ? шт./s]

✅ Успешно удалено файлов: 24 (Пакетное удаление)


## Describing results of test labeling on Human signal

Among 200 DB candidates were 87 TP cases (43.5%)
Among 200 DT candidates were 73 TP cases (36.5%)

As a consequance expected number of dataset for training would contain 2215 * 0.365 = 808 DT candidates and 2466 * 0.435 = 1072 DB candidates

# Main labeling (with DataBase)

## Pushing all candidates in DB for next labeling

In [ ]:
s3_client = boto3.client(
    service_name='s3',
    endpoint_url='https://storage.yandexcloud.net',
    aws_access_key_id=ACCESS_KEY_ID,
    aws_secret_access_key=SECRET_ACCESS_KEY
)

In [ ]:
upload_heuristic_sample_to_s3(
    labeled_dict=clear_labeled_data,
    s3_client=s3_client,
    bucket_name=YANDEX_S3_BUCKET,
    s3_folder=main_dataset_prefix,
    target_type=[TARGET_MAP["DB"], TARGET_MAP["DT"]],
    sample_size="all",
    max_workers=10
)

☁️ Подготовка [Double Bottom & Double Top] к загрузке в S3 (Бакет: tickframe-candidates, Папка: labeled_train_v1)
📊 Выбрана загрузка АБСОЛЮТНО ВСЕХ кандидатов (4681 шт.)...


Uploading Patterns:   0%|          | 0/4681 [00:00<?, ?it/s]

✅ Готово! Успешно загружено 4681 картинок в S3.


## Clear all candidates from DB

In [ ]:
clear_s3_folder_fast(s3_client, YANDEX_S3_BUCKET, main_dataset_prefix)

🗑️ Начинаем очистку папки s3://tickframe-candidates/labeled_train_v1/ ...


  ↳ Удаление файлов: 0 шт. [00:00, ? шт./s]

✅ Успешно удалено файлов: 43 (Пакетное удаление)


# Training model

## Creating dataset for training

In [14]:
def encode_labels(label):
    if pd.isna(label):
        return -1

    label_str = str(label)

    if "Double Top" in label_str or "DT" in label_str:
        return TARGET_MAP['DT']
    elif "Double Bottom" in label_str or "DB" in label_str:
        return TARGET_MAP["DB"]
    else:
        return 0

In [15]:
# Pooling labeled data from google drive
labeled_DT_DB_path = '/content/drive/MyDrive/Crypto_Labeled_Data/labeled_DT_DB.csv'

labeled_DT_DB = pd.read_csv(labeled_DT_DB_path, encoding='utf-8')
labeled_DT_DB['Target'] = labeled_DT_DB['label'].apply(encode_labels)

# Pooling RAW data from google drive


In [16]:
labeled_DT_DB.head()

,agreement,annotation_id,annotator,created_at,id,image,label,lead_time,updated_at,Target
0,100.0,98675095.0,amirgaf29@gmail.com,2026-07-03T13:53:42.204443Z,273780490,s3://tickframe-candidates/labeled_train_v1/DB_...,Шум / Флэт,63.530,2026-07-03T13:53:42.204457Z,0
1,100.0,98675117.0,amirgaf29@gmail.com,2026-07-03T13:53:57.970569Z,273780494,s3://tickframe-candidates/labeled_train_v1/DB_...,Double Bottom (W),14.031,2026-07-03T13:53:57.970581Z,2
2,100.0,98675129.0,amirgaf29@gmail.com,2026-07-03T13:54:05.605327Z,273780496,s3://tickframe-candidates/labeled_train_v1/DB_...,Double Bottom (W),5.270,2026-07-03T13:54:05.605338Z,2
3,100.0,98675140.0,amirgaf29@gmail.com,2026-07-03T13:54:15.261268Z,273780498,s3://tickframe-candidates/labeled_train_v1/DB_...,Double Bottom (W),8.223,2026-07-03T13:54:15.261281Z,2
4,100.0,98675151.0,amirgaf29@gmail.com,2026-07-03T13:54:24.408233Z,273780501,s3://tickframe-candidates/labeled_train_v1/DB_...,Шум / Флэт,7.549,2026-07-03T13:54:24.408247Z,0


In [ ]:
def parse_s3_image_url(image_url: str):
    """
    Парсит URL картинки из Label Studio и возвращает торговую пару и индекс (idx) свечи.
    Пример: 's3://tickframe-candidates/labeled_train_v1/DB_idx_10022_AVAX_USDT.png'
    Результат: ('AVAX/USDT', 10022)
    """
    if pd.isna(image_url):
        return None, None

    base_name = os.path.basename(str(image_url))
    name_without_ext, _ = os.path.splitext(base_name)

    if "_idx_" not in name_without_ext:
        return None, None

    parts = name_without_ext.split("_idx_")
    right_part = parts[1]

    idx_str, safe_symbol = right_part.split('_', 1)
    idx = int(idx_str)

    if "_USDT" in safe_symbol:
        symbol = safe_symbol.replace("_USDT", "/USDT")
    else:
        symbol = safe_symbol.replace("_", "/", 1)

    return symbol, idx